### Установка зависимостей

In [ ]:
pip install python-dotenv>=1.0.0
pip install httpx>=0.27.0
pip install mlflow>=2.14.0
pip install pandas>=2.2.0
pip install langchain-chroma>=0.1.0
pip install langchain-core>=0.2.0
pip install langchain-community>=0.2.0
pip install langchain-gigachat>=0.1.0
pip install langchain-huggingface>=0.1.0
pip install langchain-text-splitters>=0.2.0
pip install nltk>=3.9.4
pip install pymorphy3>=2.0.6
pip installjson-repair>=0.64.5

### Импорты

In [1]:
import os
import json
import re
import sqlite3
import pandas as pd

import prompts
import utils

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from datetime import datetime

from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.retrievers import BM25Retriever
from langchain_gigachat.chat_models import GigaChat
from langchain_huggingface import HuggingFaceEmbeddings

/tmp/ipykernel_42493/2896073508.py:21: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [2]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /home/vadim/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Входные данные

In [134]:
user_query = "Я ищу квартиру в проекте ПИК класса комфорт плюс с надземным паркингом"

In [107]:
right_answer = "['Онежский вал', 'Кавказкий бульвар 51', 'Бусиновский парк', 'Лосиноостровский парк']"

In [4]:
gigachat_ak = "Your GigaChat APIkey"

In [6]:
weights = [0.5, 0.5]

In [71]:
CHUNK_SIZE = 700
LLM = "GigaChat-2"
EMBED_MODEL = "Qwen/Qwen3-Embedding-0.6B"
#EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

### Создание таблицы projects

In [25]:
os.makedirs("./db/mos_estate", exist_ok=True)
conn = sqlite3.connect(r'./db/mos_estate/mos_estate.sqlite3')
cursor = conn.cursor()

In [26]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS projects (
    id INTEGER PRIMARY KEY,
    developer TEXT NOT NULL,
    project TEXT NOT NULL,
    class TEXT,
    district TEXT,
    metro TEXT,
    dist_to_metro REAL,
    num_residents INTEGER,
    shcool_plan INTEGER,
    shcool_fact INTEGER,
    nursery_plan INTEGER,
    nursery_fact INTEGER,
    parking TEXT
)
''')

### Заполнение таблицы projects

In [27]:
data_to_insert = []

In [28]:
with open(r'./raw_data/general_data.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line.strip())
        row = (
            record.get('developer'),
            record.get('project'),
            record.get('class'),
            record.get('district'),
            record.get('metro'),
            record.get('dist_to_metro'),
            record.get('num_residents'),
            record.get('shcool_plan'),
            record.get('shcool_fact'),
            record.get('nursery_plan'),
            record.get('nursery_fact'),
            record.get('parking')
        )
        data_to_insert.append(row)

In [29]:
try:
    cursor.execute('BEGIN TRANSACTION')
    
    cursor.executemany('''
        INSERT INTO projects (
            developer, project, class, district, metro, dist_to_metro,
            num_residents, shcool_plan, shcool_fact, nursery_plan, nursery_fact, parking
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', data_to_insert)
    
    conn.commit()
    print(f"Успешно вставлено {len(data_to_insert)} записей")
    
except Exception as e:
    conn.rollback()
    print(f"Ошибка при вставке: {e}")

finally:

    cursor.execute('PRAGMA synchronous = NORMAL')
    cursor.execute('PRAGMA journal_mode = DELETE')

Успешно вставлено 96 записей


### Создание таблицы descriptions

In [30]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS descriptions (
    id INTEGER PRIMARY KEY,
    id_project INTEGER NOT NULL,
    description TEXT
)
''')

### Заполнение таблицы description

In [31]:
data_to_insert = []

In [32]:
with open(r'./raw_data/descriptions.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line.strip())
        row = (
            record.get('id_project'),
            record.get('description')
        )
        data_to_insert.append(row)

In [33]:
try:
    cursor.execute('BEGIN TRANSACTION')
    
    cursor.executemany('''
        INSERT INTO descriptions (
            id_project, description
        ) VALUES (?, ?)
    ''', data_to_insert)
    
    conn.commit()
    print(f"Успешно вставлено {len(data_to_insert)} записей")
    
except Exception as e:
    conn.rollback()
    print(f"Ошибка при вставке: {e}")

finally:
    cursor.execute('PRAGMA synchronous = NORMAL')
    cursor.execute('PRAGMA journal_mode = DELETE')

Успешно вставлено 96 записей


In [34]:
conn.close()

### Очистка данных

In [35]:
conn = sqlite3.connect(r'./db/mos_estate/mos_estate.sqlite3')
cursor = conn.cursor()

In [36]:
df = pd.read_sql_query("SELECT * FROM descriptions", conn)

In [37]:
def text_cleaner(text):

    # Конвертация в str
    text = str(text)
    
    # Преобразование текста в нижний регистр
    text = text.lower()

    # Удаление лишних пробелов
    #text = re.sub(r'\s+', ' ', text)

    # Удаление знаков препинания
    text = re.sub(r'[^\w\s]', '', text)

    # Удаление стоп-слов
    stop_words = set(stopwords.words('russian'))
    tokens = word_tokenize(text)
    filtered_text = [word for word in tokens if word.lower() not in stop_words]
    
    # Объединение лемматизированных слов обратно в текст
    cleaned_text = ' '.join(filtered_text)

    return cleaned_text

In [38]:
df['cleaned_description'] = df['description'].map(lambda description: text_cleaner(description))

In [39]:
df.head()

,id,id_project,description,cleaned_description
0,1,1,«Ильинские луга» — масштабный проект с собстве...,ильинские луга масштабный проект собственной и...
1,2,2,«Белая Дача парк» — проект в Московской област...,белая дача парк проект московской области горо...
2,3,3,Жилой район «Саларьево парк» расположен в Ново...,жилой район саларьево парк расположен новой мо...
3,4,4,Жилой квартал «Люблинский парк» расположен на ...,жилой квартал люблинский парк расположен югово...
4,5,5,«Зелёный парк» — масштабный проект в экологичн...,зелёный парк масштабный проект экологичном рай...


### Запись очищенных данных в бд

In [40]:
cursor.execute('ALTER TABLE descriptions ADD COLUMN cleaned_description TEXT')
conn.commit()

In [41]:
rows = list(zip(df['cleaned_description'], df['id_project']))

In [42]:
update_query = '''
UPDATE descriptions
SET cleaned_description = ?
WHERE id_project = ?
'''

In [43]:
cursor.executemany(update_query, rows)
conn.commit()
print(f"Обновлено строк: {cursor.rowcount}")

Обновлено строк: 96


In [44]:
conn.commit()

In [45]:
conn.close()

### Подготовка чанков

In [46]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [47]:
conn = sqlite3.connect(r'./db/mos_estate/mos_estate.sqlite3')
cursor = conn.cursor()

cursor.execute(
    """  SELECT
    p.developer,
    p.project,
    p.class,
    p.district,
    p.metro,
    p.dist_to_metro,
    p.num_residents,
    p.shcool_plan,
    p.shcool_fact,
    p.nursery_plan,
    p.nursery_fact,
    p.parking,
    d.cleaned_description
        FROM projects AS p
        JOIN descriptions AS d
        ON p.id = d.id_project;
"""
)
all_rows = cursor.fetchall()
conn.close()

In [48]:
all_docs = []
for row in all_rows:
    content = row[12]
    meta = {
        'dev': row[0],
        'proj': row[1],
        'h_class': row[2], 
        'distr': row[3], 
        'metro': row[4],
        'dist_to_metro': row[5],
        'num_resid': row[6],
        's_plan': row[7],
        's_fact': row[8],
        'n_plan': row[9],
        'n_fact': row[10],
        'parking': row[11],
    }
    all_docs.append(Document(page_content=content, metadata=meta))

In [49]:
len(all_docs)

96

In [50]:
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=50)
chunked_docs = splitter.split_documents(all_docs)

In [51]:
print(f"Было документов: {len(all_docs)}, стало чанков: {len(chunked_docs )}")

Было документов: 96, стало чанков: 365


### Создание векторной базы

In [52]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

In [53]:
persist_directory = "./db/chroma"
os.makedirs(persist_directory, exist_ok=True)

In [54]:
postfix = ""
if EMBED_MODEL == "Qwen/Qwen3-Embedding-0.6B":
    postfix = "_qwen"
if EMBED_MODEL == "sentence-transformers/all-MiniLM-L6-v2":
    postfix = "_se"
collection_name = "chunk_" + str(CHUNK_SIZE) + postfix

In [55]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={
        "device": "cpu",
    },
    encode_kwargs={
        "normalize_embeddings": True,
    },
)
vectorstore = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name=collection_name
)

I0000 00:00:1793956865.979279    9411 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 91e26af2-6aa4-459a-904b-bc70bd18aa52)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-Embedding-0.6B/97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3/config.json
Retrying in 1s [Retry 1/5].


### Проверка векторной базы

In [57]:
import chromadb

client = chromadb.PersistentClient(path=persist_directory)
print(client.list_collections())

[Collection(name=chunk_700_qwen)]


In [58]:
import chromadb
client = chromadb.PersistentClient(path=persist_directory)
col = client.get_collection(collection_name)
print(col.count())  # должно быть > 0

365


### Формирование цепочки выделения фильтра к векторной базе

In [59]:
json_parser = JsonOutputParser()

In [89]:
llm = GigaChat(
            credentials=gigachat_ak,
            model=LLM,
            verify_ssl_certs=False,
        )

In [90]:
metadata_prompt = ChatPromptTemplate.from_messages([
    ("system", prompts.metadata_system),
    ("human", prompts.metadata_human),
], template_format="jinja2")

In [91]:
extractor_chain = (
    RunnablePassthrough()
    | metadata_prompt
    | llm
    | json_parser
)

### Выделение фильтра метаданных

In [135]:
# Получаем фильтр из вопроса
raw_filter = utils.safe_llm_invoke(extractor_chain, {"question": user_query})
raw_filter

{'dev': 'ПИК', 'h_class': 'Комфорт плюс', 'parking': 'Надземный'}

In [136]:
response = extractor_chain.invoke({"question": user_query})

In [137]:
# Парсим фильтр в словарь
parsed_filter = utils.parse_filter(raw_filter)
parsed_filter

{'dev': 'ПИК', 'h_class': 'Комфорт плюс', 'parking': 'Надземный'}

In [138]:
# Нормализуем фильтр под требования Chroma
normalized_filter = utils.normalize_chroma_filter(parsed_filter)
normalized_filter

{'$and': [{'dev': {'$eq': 'ПИК'}},
  {'h_class': {'$eq': 'Комфорт плюс'}},
  {'parking': {'$eq': 'Надземный'}}]}

### Готовим BM25 ретривер

In [139]:
all_docs_raw = vectorstore.get()

In [140]:
docs_docs = all_docs_raw["documents"]
docs_meta = all_docs_raw["metadatas"]

In [141]:
all_documents = [
            Document(page_content=d, metadata=m)
            for d, m in zip(docs_docs, docs_meta)
]

In [142]:
bm25_retriever = BM25Retriever.from_documents(all_documents, 
                                              preprocess_func=utils.russian_preprocess,
                                              k=10)

### Готовим векторный ретривер

In [143]:
search_kwargs = {"k": 10}
if normalized_filter is not None:
    search_kwargs["filter"] = normalized_filter
    print(f"[RAG] Поиск с фильтром {normalized_filter}")
else:
    print("[RAG] Фильтр не извлечён или пуст → поиск без фильтра")
vector_retriever = vectorstore.as_retriever(search_kwargs=search_kwargs)
docs = vector_retriever.invoke(user_query)
print(f"[RAG] Найдено документов векторным поиском: {len(docs)}")
if normalized_filter and not docs:
    print("[RAG] Поиск с фильтром по запросу вернул пустоту → повторный поиск без фильтра")
    search_kwargs = {"k": 10}
    vector_retriever = vectorstore.as_retriever(search_kwargs=search_kwargs)
    docs = vector_retriever.invoke(user_question)
    print(f"[RAG] Повторный поиск без фильтра → найдено: {len(docs)}")

[RAG] Поиск с фильтром {'$and': [{'dev': {'$eq': 'ПИК'}}, {'h_class': {'$eq': 'Комфорт плюс'}}, {'parking': {'$eq': 'Надземный'}}]}
[RAG] Найдено документов векторным поиском: 10


### Получение ответов от ретриверов

In [144]:
retrievers = [bm25_retriever, vector_retriever]
docs = utils.ensemble_retrieve(user_query, retrievers, weights=weights, const_rrf=60, k=5)

### Чанки в контекст

In [145]:
docs

[Document(id='521c27ea-4337-409c-98d2-ce436feb13ba', metadata={'n_fact': 0, 'distr': 'Западное Дегунино\xa0', 'proj': 'Онежский вал', 'metro': 'Селигерская', 's_plan': 800, 'h_class': 'Комфорт плюс', 'n_plan': 350, 'dist_to_metro': 1.1, 'num_resid': 3500, 'dev': 'ПИК', 's_fact': 0, 'parking': 'Надземный'}, page_content='свежем воздухе дворах будут ждать зелёные парки местами отдыха детскими спортивными площадками первых этажах домов торговая аллея магазинами кафе полезными сервисами владельцев автомобилей построим двухуровневый подземный наземных паркинга школы отличием школы проектах пик созданы всестороннего развития ребёнка рядом домом светлые классы просторные атриумы дают пространство учёбы отдыха творчества итполигоны робоклассы лаборатории инновационные программы образования ведущих вузов компаний помогают раскрыть способности развить практические навыки планировки онежском валу широкий выбор планировок мастерспальнями собственной ванной просторными кухнямигостиными компактными 

###  Формируем контекст, готовим промпт и вызываем LLM

In [146]:
context_str = utils.get_context(docs)

In [147]:
base_prompt = ChatPromptTemplate.from_messages([
    ('system', prompts.system_prompt_2),
    ('human', prompts.human_prompt)
], template_format="jinja2")

In [148]:
messages = base_prompt.format_messages(context=context_str, question=user_query)

### Ответ LLM

In [149]:
if docs:
    response = llm.invoke(messages)
    print(response.content)
else:
    print("Не знаю")

Из представленных проектов ПИК класса "Комфорт плюс" с надземным паркингом подходят следующие:

1. Онежский вал
2. Бусиновский парк
3. Кавказкий бульвар 51

Ответ: Онежский вал, Бусиновский парк, Кавказкий бульвар 51


### Автоматический расчет метрики на тестовом наборе вопросов

In [150]:
questions = []

In [151]:
# Короткая версия вопросов (5 вопросов)
#test_path = "./raw_data/questions_lite.jsonl"

In [152]:
# Полная версия вопросов (86 вопросов)
test_path = "./raw_data/questions_full.jsonl"

In [153]:
with open(test_path, 'r', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line)
        data = dict(record)
        questions.append(data)

In [154]:
len(questions)

86

In [155]:
def rag_with_filter(user_question: str) -> str:
    # 1. Получаем фильтр из вопроса
    raw_filter = utils.safe_llm_invoke(extractor_chain, {"question": user_query})
    print("[DEBUG] raw filter from LLM:", raw_filter, "type:", type(raw_filter))

    # 2. Парсим фильтр в словарь
    parsed_filter = utils.parse_filter(raw_filter)
    print("[DEBUG] parsed filter from LLM:", parsed_filter, "type:", type(parsed_filter))

    # 3. Нормализуем фильтр под требования Chroma
    normalized_filter = utils.normalize_chroma_filter(parsed_filter)
    print("[DEBUG] normalized filter:", normalized_filter, "type:", type(normalized_filter))

    # 4. Создаём ретривер для векторного поиска
    search_kwargs = {"k": 10}
    if normalized_filter is not None:
        search_kwargs["filter"] = normalized_filter
        print(f"[RAG] Поиск с фильтром {normalized_filter}")
    else:
        print("[RAG] Фильтр не извлечён или пуст → поиск без фильтра")

    vector_retriever = vectorstore.as_retriever(search_kwargs=search_kwargs)
    docs = vector_retriever.invoke(user_question)
    print(f"[RAG] Найдено документов векторным поиском: {len(docs)}")

    # 5. Если ретривер векторного поиска с фильтром ничего не вернул, то проводим повторный поиск без фильтра
    if normalized_filter and not docs:
        print("[RAG] Поиск с фильтром по запросу вернул пустоту → повторный поиск без фильтра")
        search_kwargs = {"k": 10}
        vector_retriever = vectorstore.as_retriever(search_kwargs=search_kwargs)
        docs = vector_retriever.invoke(user_question)
        print(f"[RAG] Повторный поиск без фильтра → найдено: {len(docs)}")

    # 6. Получение ответов от ретриверов
    retrievers = [bm25_retriever, vector_retriever]
    docs = utils.ensemble_retrieve(user_question, retrievers, weights=weights, const_rrf=60, k=5)

    # 7. Если и после этого пусто — не отправляем в LLM
    if not docs:
        return "Не знаю"

    # 8. Формируем контекст
    context_str = utils.get_context(docs)

    # 9. Готовим промпт и вызываем LLM
    messages = base_prompt.format_messages(context=context_str, question=user_question)
    response = utils.safe_llm_invoke(llm, messages)

    return response.content, context_str

In [156]:
for num, question in enumerate(questions):
    user_query = question['question']
    print(f"[DEBUG] user query: {user_query}")
    answer_rag, docs = rag_with_filter(user_query)
    if answer_rag is None:
        answer_rag = "Не удалось получить ответ от языковой модели из-за временной ошибки соединения."
    print(f"[ANSWER] answer_rag: {answer_rag}")
    question['id'] = num
    question['answer_rag'] = utils.truncate_before_answer(answer_rag)
    question['chunks'] = docs

[DEBUG] user query: Какое плановое количество жителей указано для проекта «Ильинские луга»?
[DEBUG] raw filter from LLM: {'proj': 'Ильинские луга', 'num_resid': {'$exists': True}} type: <class 'dict'>
[DEBUG] parsed filter from LLM: {'proj': 'Ильинские луга', 'num_resid': {'$exists': True}} type: <class 'dict'>
[DEBUG] normalized filter: {'proj': {'$eq': 'Ильинские луга'}} type: <class 'dict'>
[RAG] Поиск с фильтром {'proj': {'$eq': 'Ильинские луга'}}
[RAG] Найдено документов векторным поиском: 4
[ANSWER] answer_rag: Ответ: Планируемое количество жителей для проекта «Ильинские луга» составляет 15000 человек.
[DEBUG] user query: Каково расстояние до метро для проекта «Саларьево парк» (в км)?
[DEBUG] raw filter from LLM: {'proj': 'Саларьево парк'} type: <class 'dict'>
[DEBUG] parsed filter from LLM: {'proj': 'Саларьево парк'} type: <class 'dict'>
[DEBUG] normalized filter: {'proj': {'$eq': 'Саларьево парк'}} type: <class 'dict'>
[RAG] Поиск с фильтром {'proj': {'$eq': 'Саларьево парк'}}


In [157]:
question_df = pd.DataFrame(questions)

In [158]:
answers = [{
    'id': el['id'],
    'answer': el['answer'],
    'answer_rag': el['answer_rag']
} for el in questions]

In [159]:
batch_size = 50
metrics_dfs = []

In [160]:
metrics_prompt = ChatPromptTemplate.from_messages([
    ('system', prompts.sys_metrics_prompt_2),
    ('human', prompts.hum_metrics_prompt)
], template_format="jinja2")

In [161]:
for i in range(0, len(answers), batch_size):
    batch = answers[i:i + batch_size]
    if not batch:
        continue
    batch_metrics = utils.get_metrics_from_llm(metrics_prompt, llm, batch)
    if batch_metrics:
        metrics_dfs.append(pd.DataFrame(batch_metrics))

In [162]:
if metrics_dfs:
    metrics_df = pd.concat(metrics_dfs, axis=0, ignore_index=True)
else:
    metrics_df = pd.DataFrame(columns=["id", "correct"])

In [163]:
result_df = utils.safe_merge_metrics(question_df, metrics_df)

In [164]:
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
os.makedirs("./outputs", exist_ok=True)
csv_filename = f"base_result_{timestamp}.csv"
csv_dir = "./outputs/" + csv_filename 
metrics_filename = f"metrics_result_{timestamp}.csv"
metrics_dir = "./outputs/" + metrics_filename

In [165]:
metrics_df.to_csv(metrics_dir)
result_df.to_csv(csv_dir)

In [166]:
accuracy = result_df['correct'].sum() / len(result_df['correct'])

### Точность на тестовом наборе вопросов

In [167]:
accuracy

0.8488372093023255